# Ch.1 — Logistic Regression

> **The story.** In **1838**, Belgian mathematician **Pierre-François Verhulst** was wrestling with Malthus's prediction of exponential population growth — the census data disagreed. His answer was an S-shaped curve, $P(t) = \frac{K}{1 + e^{-r(t - t_0)}}$, which he named the _logistic function_ (from Greek _logistikos_, "skilled in calculation"). It sat in a drawer for a century. In **1944**, biostatistician **Joseph Berkson** needed to model the probability that a patient responds to a drug dose — probabilities that must stay between 0 and 1 while linear models drift outside that range. He recognised Verhulst's S-curve as the ideal probability squasher and coined the word _logit_. **David Cox** completed the picture in **1958**, placing logistic regression on firm statistical ground via maximum likelihood estimation under Bernoulli outcomes. When the neural network renaissance came, researchers realised that a single neuron with sigmoid activation and cross-entropy loss _is_ logistic regression — and everything before was a special case.
>
> **Where you are.** Topic 01 (Regression) taught you to predict continuous values. The FaceAI product team now needs a binary signal: is this person **Smiling** or not? Logistic regression reuses $z = \mathbf{w} \cdot \mathbf{x} + b$ from linear regression, adds a sigmoid at the output, and swaps MSE for the statistically principled binary cross-entropy loss. This chapter sets the **~88% accuracy baseline** that all subsequent classifiers must beat on the way to 90%.
>
> **Notation in this chapter.** $\mathbf{x} \in \mathbb{R}^d$ — input feature vector ($d=200$ HOG-like features); $y \in \{0,1\}$ — true label (1 = Smiling); $z = \mathbf{w} \cdot \mathbf{x} + b$ — the **logit** (raw linear output); $\hat{p} = \sigma(z) = 1/(1+e^{-z})$ — **sigmoid** output (predicted probability); $L = -\frac{1}{N}\sum_i[y_i \log\hat{p}_i + (1-y_i)\log(1-\hat{p}_i)]$ — **binary cross-entropy** (BCE) loss; $TP, FP, TN, FN$ — confusion-matrix counts.

---

## 0 · The Challenge

> **The mission**: Launch **FaceAI** — auto-tag 202,599 celebrity photos across 40 binary attributes with >90% average accuracy.
>
> | #   | Constraint       | Target                           | Status        |
> | --- | ---------------- | -------------------------------- | ------------- |
> | 1   | ACCURACY         | >90% avg across 40 attributes    | Starting — 0% |
> | 2   | GENERALIZATION   | Unseen celebrity faces           |               |
> | 3   | MULTI-LABEL      | 40 simultaneous attributes       |               |
> | 4   | INTERPRETABILITY | Which features drive predictions |               |
> | 5   | PRODUCTION       | <200ms inference                 |               |

**What we know so far:**

- Topic 01: Linear regression — $\hat{y} = \mathbf{w} \cdot \mathbf{x} + b$, MSE loss, gradient descent
- **But we can only predict continuous values, not probabilities or class labels.**

**What's blocking us:**
FaceAI needs binary labels: _is this person Smiling?_ Linear regression outputs unbounded real numbers — not probabilities. A raw output of $3.8$ has no probabilistic meaning. When you threshold ("anything above 2.5 is Smiling"), the threshold is arbitrary and MSE sends the wrong gradient signal for binary targets.

**What this chapter unlocks:**

- **Sigmoid activation** — squash $z \in (-\infty, +\infty)$ → $\hat{p} \in (0, 1)$
- **Binary cross-entropy loss** — the statistically principled loss for probability predictions
- **Confusion matrix** — first look at TP/FP/TN/FN (full treatment in Ch.3)
- **Constraint #1 PARTIAL** — ~88% accuracy on Smiling (first dent in the 90% target)

```mermaid
graph LR
 A["Topic 01: Regression<br/>continuous → ŷ ∈ ℝ"] --> B["Ch.1: Logistic Regression<br/>binary → p̂ ∈ (0,1) "]
 B --> C["Ch.2: Classical Classifiers"]
 C --> D["Ch.3: Metrics"]
 D --> E["Ch.4: SVM"]
 E --> F["Ch.5: Tuning"]
 style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style B fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style D fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style E fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style F fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

---

**Chapter map:**

| §   | Concept                   | Where   |
| --- | ------------------------- | ------- |
| §0  | CelebA data proxy         | Cell 3  |
| §1  | Sigmoid activation        | Cell 5  |
| §2  | Training (scale + fit)    | Cell 7  |
| §3  | Confusion matrix          | Cell 9  |
| §4  | BCE vs MSE loss landscape | Cell 11 |
| §5  | ROC curve & AUC           | Cell 13 |
| §6  | Probability distribution  | Cell 15 |
| §7  | Feature weights           | Cell 17 |
| §8  | Threshold sweep           | Cell 19 |
| §9  | Summary                   | Cell 21 |


In [ ]:
# ── Setup & Imports ────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    ConfusionMatrixDisplay,
)

IMG_DIR = Path("img")
IMG_DIR.mkdir(exist_ok=True)
SAVE_KW = dict(dpi=150, bbox_inches="tight")
np.random.seed(42)
print("Imports OK")

## §0 Data — CelebA Smiling Attribute

We use a synthetic proxy that mimics CelebA's Smiling distribution (48% positive).
Replace with real CelebA via `torchvision.datasets.CelebA` when available.

**CelebA quick-start** (replace synthetic proxy):

```python
# Option 1: Kaggle mirror (jessicali9530/celeba-dataset)
# Option 2: Official CelebA — download aligned images + list_attr_celeba.txt
#
# Folder layout expected:
# data/celeba/img_align_celeba/ — face images (000001.jpg, ...)
# data/celeba/metadata/list_attr_celeba.txt
#
# Minimal loader:
# from pathlib import Path
# import pandas as pd
# attr = pd.read_csv('data/celeba/metadata/list_attr_celeba.txt',
# sep=r'\s+', skiprows=1)
# attr = (attr + 1) // 2 # {-1,+1} → {0,1}
# y_smiling = attr['Smiling'].astype(int)
# # Use official train/val/test splits to avoid leakage
# # Persist scaler/PCA/HOG settings alongside the model
```


In [ ]:
# ── Synthetic CelebA Proxy ─────────────────────────────
# Simulates HOG-like features from 64×64 face images
# 5,000 samples, 1,764 features (HOG: 8×8 cells, 2×2 blocks)
from sklearn.datasets import make_classification

N_SAMPLES = 5000
N_FEATURES = 200  # compressed HOG-like features

X, y = make_classification(
    n_samples=N_SAMPLES,
    n_features=N_FEATURES,
    n_informative=40,
    n_redundant=20,
    n_clusters_per_class=3,
    weights=[0.52, 0.48],
    flip_y=0.05,
    random_state=42,
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Smiling rate — Train: {y_train.mean():.2%}, Test: {y_test.mean():.2%}")

## §1 The Sigmoid Function

Linear regression left you with a raw score $z = \mathbf{w} \cdot \mathbf{x} + b$ that could be $-42$ or $+312$ — no meaning as a probability. The sigmoid is the fix: it squashes any real number into $(0, 1)$, producing a genuine probability. Without it, gradient descent optimises a loss that doesn't match what you care about, and any threshold you pick is a guess.

```
Sigmoid: σ(z) = 1 / (1 + e^(-z))

σ(z)
1.0 ─────────────────────────────────────────────
0.9 ···················╱·························
0.7 ··················╱··························
0.5 ─────────────────╫─────────── threshold = 0.5
0.3 ·················╱···························
0.1 ·················╱···························
0.0 ─────────────────────────────────────────────
     -6    -4    -2    0    +2    +4    +6     z
                        ↑
                   Decision boundary
                   (predict Smiling if σ(z) ≥ 0.5)
```

> **Optional depth:** The formula $\sigma(z) = \frac{1}{1+e^{-z}}$ derives from maximising the Bernoulli log-likelihood $\sum_i [y_i \log\hat{p}_i + (1-y_i)\log(1-\hat{p}_i)]$ — the same expression that becomes binary cross-entropy when negated. The sigmoid is not a design choice; it is the _unique_ function that satisfies the Bernoulli MLE.


In [ ]:
# ── Sigmoid Visualization ──────────────────────────────
z = np.linspace(-6, 6, 300)
sigma = 1 / (1 + np.exp(-z))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(z, sigma, "b-", linewidth=2)
ax.axhline(0.5, color="gray", linestyle="--", alpha=0.5)
ax.axvline(0, color="gray", linestyle="--", alpha=0.5)
ax.set_xlabel("Logit z = w·x + b")
ax.set_ylabel("σ(z) = P(Smiling)")
ax.set_title("Sigmoid Function: Logit → Probability")
ax.annotate(
    "Threshold at 0.5",
    xy=(0, 0.5),
    xytext=(2, 0.3),
    arrowprops=dict(arrowstyle="->", color="red"),
    fontsize=10,
    color="red",
)
fig.savefig(IMG_DIR / "sigmoid.png", **SAVE_KW)
plt.show()

```mermaid
flowchart LR
    A["Face image\n64×64 pixels"] -->|"HOG\ndescriptors"| B["Feature vector\nx ∈ ℝ^200"]
    B -->|"z = w·x + b"| C["Logit z\n∈ (-∞, +∞)"]
    C -->|"σ(z)"| D["Probability p̂\n∈ (0, 1)"]
    D -->|"p̂ ≥ 0.5 ?"| E{{"Threshold"}}
    E -->|"Yes"| F["Smiling"]
    E -->|"No"| G["Not Smiling"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style G fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

### What §1 established — and what it still doesn't solve

The sigmoid turns a raw linear score into a probability — FaceAI can now say "73% confident this face is Smiling" instead of "score = 2.3." The classification pipeline is complete end-to-end: HOG features → linear score → sigmoid → probability → threshold → label. What it doesn't solve: we've defined the forward pass but haven't found weights $\mathbf{w}$ yet. Training is §2.


## §2 Training Logistic Regression

The sigmoid defines the forward pass. Now find weights $\mathbf{w}$ that produce probabilities close to the true labels — and **scale the features first**. HOG descriptor magnitudes vary by a factor of 100 across channels; unscaled features make gradient descent converge 1,000× slower. Scikit-learn's `lbfgs` solver then minimises binary cross-entropy directly, returning trained weights in under a second for 4,000 examples.


In [ ]:
# ── Scale + Train ─────────────────────────────────────
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

model = LogisticRegression(C=1.0, max_iter=500, solver="lbfgs", random_state=42)
model.fit(X_train_s, y_train)

train_acc = model.score(X_train_s, y_train)
test_acc = model.score(X_test_s, y_test)
print(f"Train accuracy: {train_acc:.3f}")
print(f"Test accuracy:  {test_acc:.3f}")

## §3 Confusion Matrix

Accuracy is one number: "87% correct." But that single number hides whether the 13% errors are spread evenly or concentrated on one class. The confusion matrix shows every prediction outcome as a 2×2 grid — and for imbalanced attributes (Bald at 2.2% in the full CelebA task), the four cells tell completely different stories.

```
           Predicted
           Not Smiling  |  Smiling
         ┌─────────────┼────────────┐
Actual   │      TN      │     FP     │  ← "False alarm"
Not      ├─────────────┼────────────┤
Smiling  │      FN      │     TP     │  ← "Hit"
Smiling  └─────────────┴────────────┘
              ↑
         "Miss" — the error that costs
         FaceAI a misclassified face
```


In [ ]:
# ── Confusion Matrix ───────────────────────────────────
y_pred = model.predict(X_test_s)

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=["Not Smiling", "Smiling"], cmap="Blues", ax=ax
)
ax.set_title("Confusion Matrix — Smiling Detection")
fig.savefig(IMG_DIR / "confusion_matrix.png", **SAVE_KW)
plt.show()

print(classification_report(y_test, y_pred, target_names=["Not Smiling", "Smiling"]))

### What §3 established — and what it still doesn't solve

The confusion matrix shows exactly which errors the model makes — actual counts of TN/FP/FN/TP. You can now answer "how many Smiling faces did we miss?" But this is the confusion matrix at threshold 0.5 — a single operating point. What if you lower the threshold to catch more Smiling faces? You trade precision for recall. That trade-off, and how to pick the right operating point, is what §5 (ROC) and §8 (threshold sweep) formalise.


## §4 Binary Cross-Entropy Loss

You have sigmoid outputs — why not just use MSE? Because MSE + sigmoid creates a non-convex loss landscape where gradients vanish whenever the model is confidently wrong. The model stalls before correcting the worst errors. BCE avoids this: its gradient is $\hat{p} - y$ — always informative, never zero except at the correct prediction. This is why Cox (1958) derived BCE from the Bernoulli likelihood rather than borrowing MSE from linear regression.

The two panels below show the difference: MSE has flat regions at both extremes (zero gradient = stuck). BCE is convex with a gradient that grows as confidence increases.


In [ ]:
# ── MSE vs BCE Loss Landscape — side-by-side comparison ──────────────────
p_hat = np.linspace(0.01, 0.99, 200)
z = np.linspace(-6, 6, 200)
sigma = 1 / (1 + np.exp(-z))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(
    "MSE vs Binary Cross-Entropy — Same Data, Different Loss Landscapes", fontsize=13
)

# ── Panel A: MSE loss (y=1) with sigmoid ───────────────────────────────
mse_loss = (1 - sigma) ** 2  # y=1
axes[0].plot(z, mse_loss, "b-", linewidth=2, label="MSE when y=1")
axes[0].set_xlabel("Logit z = w·x + b", fontsize=11)
axes[0].set_ylabel("Loss", fontsize=11)
axes[0].set_title(
    "Panel A — MSE + Sigmoid\n(non-convex, vanishing gradient)", fontsize=10
)
axes[0].annotate(
    "Gradient ≈ 0 here\n(vanishing)", xy=(-5, 0.95), fontsize=9, color="red"
)
axes[0].annotate(
    "Gradient ≈ 0 here\n(vanishing)",
    xy=(3, 0.02),
    fontsize=9,
    color="red",
    xytext=(3, 0.15),
    arrowprops=dict(arrowstyle="->", color="red"),
)
axes[0].legend()
axes[0].set_ylim(0, 1.1)

# ── Panel B: BCE loss (y=1) ────────────────────────────────────────────
bce_loss = -np.log(sigma)  # y=1: -log(p̂)
axes[1].plot(z, bce_loss, "r-", linewidth=2, label="BCE when y=1")
axes[1].set_xlabel("Logit z = w·x + b", fontsize=11)
axes[1].set_ylabel("Loss", fontsize=11)
axes[1].set_title("Panel B — BCE + Sigmoid\n(convex, gradient = p̂ − y)", fontsize=10)
axes[1].annotate(
    "Large penalty\nfor confident wrong",
    xy=(-4, 4.5),
    fontsize=9,
    xytext=(-2, 5.5),
    arrowprops=dict(arrowstyle="->", color="darkred"),
    color="darkred",
)
axes[1].legend()
axes[1].set_ylim(0, 7)

fig.tight_layout()
fig.savefig(IMG_DIR / "mse_vs_bce_loss.png", **SAVE_KW)
plt.show()
print(
    "Key difference: MSE gradient vanishes at extremes. BCE gradient = (p̂ - y) — always informative."
)

### What §4 established — and what it still doesn't solve

BCE gives gradient descent a loss landscape it can navigate: convex, with a gradient that is always informative. MSE + sigmoid has flat regions where $\nabla L \approx 0$ — the model stalls on its worst errors. What it doesn't solve: you now have a trained model producing probabilities, but how do you measure performance threshold-independently? A confusion matrix at 0.5 is one point. §5 introduces AUC-ROC — the full picture.


## §5 ROC Curve

The confusion matrix at threshold 0.5 is a single point. What happens at 0.3? At 0.7? The ROC curve plots True Positive Rate (recall) against False Positive Rate across every threshold — it tells you how well the model _ranks_ positives above negatives, independently of any specific operating point. For FaceAI, AUC confirms whether the Smiling model is worth deploying before you commit to any threshold.

```
TPR
1.0 ┤                    ╭──── Perfect (AUC = 1.0)
    │               ╭────╯
0.8 ┤          ╭────╯
    │      ╭───╯    ← FaceAI LogReg (AUC ≈ 0.95)
0.6 ┤    ╭─╯
    │   ╭╯
0.4 ┤  ╭╯
    │ ╭╯  ← Random baseline (AUC = 0.5)
0.2 ┤╱
    │╱
0.0 ┼──────────────────────────────
    0.0  0.2  0.4  0.6  0.8  1.0
                            FPR
```

> **Optional depth:** AUC equals $P(\hat{p}_+ > \hat{p}_-)$ — the probability that a randomly chosen positive gets a higher predicted probability than a randomly chosen negative. AUC = 0.95 means 95% of (Smiling, Not-Smiling) pairs are correctly ranked.


In [ ]:
# ── ROC Curve ──────────────────────────────────────────
y_prob = model.predict_proba(X_test_s)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, "b-", linewidth=2, label=f"LogReg (AUC={auc:.3f})")
ax.plot([0, 1], [0, 1], "k--", alpha=0.3, label="Random")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve — Smiling Detection")
ax.legend()
fig.savefig(IMG_DIR / "roc_curve.png", **SAVE_KW)
plt.show()

### What §5 established — and what it still doesn't solve

AUC-ROC gives a threshold-free summary of ranking quality — ~0.95 confirms the model genuinely separates Smiling from Not-Smiling across the full probability range, not just at 0.5. What it doesn't solve: for _severely imbalanced_ attributes (Bald at 2.2%, Mustache at 1.1%), ROC's FPR denominator is dominated by the vast negative class, making even a bad model look good. PR-AUC is more informative for those cases — that's Ch.3's contribution.


## §6 Probability Distribution

The ROC curve gives ranking quality; the probability histogram shows _separation quality_. A well-calibrated classifier produces two non-overlapping histograms — Smiling faces clustered near 1.0, Not-Smiling near 0.0. Heavy overlap near 0.5 means many borderline cases where the model is genuinely uncertain and where the 0.5 threshold may not be optimal.


In [ ]:
# ── Probability Histograms ─────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(y_prob[y_test == 0], bins=30, alpha=0.6, label="Not Smiling", color="blue")
ax.hist(y_prob[y_test == 1], bins=30, alpha=0.6, label="Smiling", color="orange")
ax.axvline(0.5, color="red", linestyle="--", label="Threshold=0.5")
ax.set_xlabel("Predicted P(Smiling)")
ax.set_ylabel("Count")
ax.set_title("Predicted Probability Distribution by True Class")
ax.legend()
fig.savefig(IMG_DIR / "prob_distribution.png", **SAVE_KW)
plt.show()

## §7 Feature Importance (Top Weights)

FaceAI's VP asked: "Which image regions drive the Smiling prediction?" Logistic regression answers directly — coefficient magnitude tells you each HOG cell's contribution to the logit. Positive weights push toward Smiling; negative weights push away. The top-20 list maps to specific 8×8 image patches: you can tell a stakeholder which facial regions are the strongest signal.


In [ ]:
# ── Top Feature Weights ────────────────────────────────
coefs = model.coef_[0]
top_k = 20
top_idx = np.argsort(np.abs(coefs))[-top_k:][::-1]

fig, ax = plt.subplots(figsize=(10, 5))
colors = ["green" if c > 0 else "red" for c in coefs[top_idx]]
ax.barh(range(top_k), coefs[top_idx], color=colors)
ax.set_yticks(range(top_k))
ax.set_yticklabels([f"HOG[{i}]" for i in top_idx])
ax.set_xlabel("Weight")
ax.set_title("Top 20 Feature Weights (green=Smiling, red=Not Smiling)")
ax.invert_yaxis()
fig.savefig(IMG_DIR / "feature_weights.png", **SAVE_KW)
plt.show()

## §8 Threshold Sweep

Default threshold 0.5 treats false positives and false negatives as equally costly. FaceAI's product context may not agree — missing a genuine Smiling face (FN) hurts user experience more than a false alarm (FP). The threshold sweep maps precision and recall at every cut-off, letting you pick the operating point that best matches the business cost of each error type, and identifies the threshold that maximises F1.


In [ ]:
# ── Threshold vs F1 Curve ──────────────────────────────
from sklearn.metrics import f1_score, precision_score, recall_score

thresholds_sweep = np.arange(0.1, 0.9, 0.02)
f1s, precs, recs = [], [], []

for t in thresholds_sweep:
    y_t = (y_prob >= t).astype(int)
    f1s.append(f1_score(y_test, y_t))
    precs.append(precision_score(y_test, y_t, zero_division=0))
    recs.append(recall_score(y_test, y_t))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(thresholds_sweep, f1s, "b-", label="F1", linewidth=2)
ax.plot(thresholds_sweep, precs, "g--", label="Precision")
ax.plot(thresholds_sweep, recs, "r--", label="Recall")
best_t = thresholds_sweep[np.argmax(f1s)]
ax.axvline(best_t, color="blue", linestyle=":", alpha=0.5)
ax.set_xlabel("Threshold")
ax.set_ylabel("Score")
ax.set_title(f"Threshold Sweep — Optimal t={best_t:.2f}")
ax.legend()
fig.savefig(IMG_DIR / "threshold_sweep.png", **SAVE_KW)
plt.show()

## §9 Summary

**Checkpoint:** FaceAI — Smiling accuracy moved from 0% (linear regression, no classification capability) → ~88% in this chapter.

This chapter set the production baseline. You converted the continuous-output linear model into a probabilistic binary classifier, introduced BCE as the principled loss, and measured performance with the confusion matrix, AUC-ROC, and threshold-tuned F1.


In [ ]:
# ── Chapter Summary ────────────────────────────────────
print("=" * 50)
print("Ch.1 — Logistic Regression for Smiling Detection")
print("=" * 50)
print(f"Test Accuracy:  {test_acc:.3f}")
print(f"ROC-AUC:        {auc:.3f}")
print(f"Best Threshold: {best_t:.2f}")
print(f"Best F1:        {max(f1s):.3f}")
print("\nConstraint #1 (ACCURACY): ~88% [Done — partial]")
print("Next: Ch.2 — Classical Classifiers (Trees, KNN, NB)")

## Coverage

**Checkpoint:** FaceAI — accuracy moved from 0% → ~88% (Smiling attribute) in this chapter.

| Concept                   | Where                  |
| ------------------------- | ---------------------- |
| Sigmoid activation        | §1                     |
| Feature scaling           | §2 (`StandardScaler`)  |
| Binary cross-entropy loss | §4 (MSE vs BCE panel)  |
| Gradient descent training | sklearn `lbfgs` solver |
| Confusion matrix          | §3                     |
| ROC-AUC                   | §5                     |
| Threshold tuning          | §8                     |

**Not covered (by design):**

- Multi-class (softmax): Topic 03 Neural Networks
- Regularisation deep-dive (L1/L2): Ch.5 Hyperparameter Tuning
- Full CelebA pipeline (202k images): replace synthetic proxy in §0
- Probability calibration (Platt scaling): sklearn `CalibrationDisplay`

**Is logistic regression enough for FaceAI?** ~88% on Smiling is 2% below the 90% target. Ch.4 (SVM) or Ch.5 (tuning) breaks 90%. Logistic regression's speed (<1ms inference) and calibrated probabilities make it the right baseline to beat.


## Exercises

1. **Regularization sweep**: Train LogReg with C ∈ {0.001, 0.01, 0.1, 1, 10, 100}. Plot train/test accuracy vs C.
2. **Multi-attribute**: Train separate LogReg models for Eyeglasses (13%) and Bald (2.5%). Compare accuracy vs F1.
3. **Feature comparison**: Compare raw pixel features (4096-dim) vs the current features. Which gives better AUC?


In [ ]:
# Exercise 1: Regularization sweep
# TODO: Train with different C values and plot accuracy curves
pass

In [ ]:
# Exercise 2: Multi-attribute classification
# TODO: Create synthetic Eyeglasses (13%) and Bald (2.5%) targets, train separate models
pass

In [ ]:
# Exercise 3: Feature comparison
# TODO: Generate raw pixel-like features (4096-dim) and compare AUC with current features
pass